In [ ]:
import numpy as np
import random as ra
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

In [ ]:
def ishigami(A, a=10., b=0.1):
    """
    Calcule la fonction d'Ishigami pour n points.

    Paramètres
    ----------
    A : ndarray de forme (3, n)
        Matrice contenant les n points (chaque colonne est un point).
    a : float
        Paramètre a de la fonction d'Ishigami.
    b : float
        Paramètre b de la fonction d'Ishigami.

    Retour
    ------
    y : ndarray de forme (n,)
        Valeurs de la fonction pour chaque point.
    """

    A = np.asarray(A)

    if A.shape[0] != 3:
        raise ValueError("La matrice A doit être de dimension (3, n).")

    x1 = A[0, :]
    x2 = A[1, :]
    x3 = A[2, :]

    y = (
        np.sin(x1)
        + a * np.sin(x2) ** 2
        + b * x3 ** 4 * np.sin(x1)
    )


    return y

In [ ]:
def pickfreeze(A,I):
    freezeAI=A[I,:];
    pickAI=np.random.uniform(-pi_value,pi_value,(2,N));
    pickfreezeA=np.insert(pickAI, I, freezeAI, axis=0)
#pickfreezeA=  np.vstack([freezeA1, pickA1]);
    return pickfreezeA

In [ ]:
#toutes les dimensions Monte Carlo, valeurs de b etc sont regroupées ici
#N est la taille de l'échantillon pour calculer la cible par un crude pick and freeze
N=1000000;
pi_value = np.pi
# Valeurs de b
# b moves from 0.4 to 2
b_vals = np.arange(0.4, 2.1, 0.1);
Nbmax=b_vals.shape[0]
#Nrpeat est le nombre d'experiences répétées à b et taille d'échantillon fixés
Nrepeat=100;
#Plage de variation et pas pour la taile des échantillons
nsamplebas=10000;
nsamplehaut=500000;
passampling=10000;
#print(b_vals)

In [ ]:
#calcul des valeurs cibles par un crude pickfreeze
S1true=np.zeros(Nbmax);
S2true=np.zeros(Nbmax);
S3true=np.zeros(Nbmax);
k=0;
for bb in b_vals:
    A=np.random.uniform(-pi_value,pi_value,(3,N));
    FA=ishigami(A,a=10.,b=bb);
    A1=pickfreeze(A,0);
    FA1=ishigami(A1,a=10.,b=bb);
    A2=pickfreeze(A,1);
    FA2=ishigami(A2,a=10.,b=bb);
    A3=pickfreeze(A,2);
    FA3=ishigami(A3,a=10.,b=bb);
    tempS1true, _ = pearsonr(FA, FA1);
#    S1true[k]=2*np.mean((FA-np.mean(FA))*(FA1))/(np.var(FA)+np.var(FA1))
    S1true[k]=tempS1true;
#    print(S1true);   http://localhost:8888/tree?token=a412e984f52a820d7a65d10d9028c99a75f31361c2fb3953
    tempS2true, _ = pearsonr(FA, FA2);
    S2true[k]=tempS2true;
#    S2true[k], _ = pearsonr(FA, FA2);
#    print(S2true);
    tempS3true, _ = pearsonr(FA, FA3);
    S3true[k]=tempS3true;
#    S3true[k], _ = pearsonr(FA, FA3);
#    print(S3true);
    k=k+1
#print(S1true[5])

In [ ]:
liste_sample=list(range(nsamplebas, nsamplehaut, passampling));
Ntrysample=int(np.trunc((nsamplehaut-nsamplebas)/passampling));
#le numéro de colonne correspond à la taille de l'échantillon dans range(nsamplebas, nsamplehaut, passampling)
SS1=np.zeros((Nrepeat,Ntrysample,Nbmax));
SS2=np.zeros((Nrepeat,Ntrysample,Nbmax));
SS3=np.zeros((Nrepeat,Ntrysample,Nbmax));
k=0;
#
for bb in b_vals:
    j=0;
    for nsample in range(nsamplebas, nsamplehaut, passampling): 
        nsamplem1=nsample-1;
        AA=[];
        rangos=[];
        FAA=[];
        FAA1=[];
        FAA1rangos=[];
        FAA2rangos=[];
        FAA3rangos=[];
#
        for i in range(Nrepeat):
            AA=np.random.uniform(-pi_value,pi_value,(3,nsample));
            FAA=ishigami(AA,a=10,b=bb);
#enumére les adresses plus petit au plus grand
            rangos=np.argsort(AA, axis=1);
#On réarrange suivant l'ordre des échantillons pour chacune des variables    
            FAA1rangos=FAA[rangos[0,:]];
            FAA2rangos=FAA[rangos[1,:]];
            FAA3rangos=FAA[rangos[2,:]];
#Calcul de l'estimateur de Chatterjee qui est l'autocorrélation des échantillons réoordonnés
            Shift_FAA1rangos=np.zeros(nsample)
            Shift_FAA1rangos[0:nsamplem1]=FAA1rangos[1:nsample];
            S1,_=pearsonr(FAA1rangos,Shift_FAA1rangos);
            SS1[i,j,k]=S1;
#    
            Shift_FAA2rangos=np.zeros(nsample)
            Shift_FAA2rangos[0:nsamplem1]=FAA2rangos[1:nsample];
            S2,_=pearsonr(FAA2rangos,Shift_FAA2rangos);
            SS2[i,j,k]=S2;
#    
            Shift_FAA3rangos=np.zeros(nsample)
            Shift_FAA3rangos[0:nsamplem1]=FAA3rangos[1:nsample];
            S3,_=pearsonr(FAA3rangos,Shift_FAA3rangos);
            SS3[i,j,k]=S3;
#
        j=j+1
    k=k+1
#print(SS1[:,:,5])

In [ ]:
#post traitement et calcul des variances asymptotiques
SSS1=SS1-S1true;
SSS2=SS2-S2true;
SSS3=SS3-S3true;
SS1square=np.multiply(SSS1,SSS1);
SS2square=np.multiply(SSS2,SSS2);
SS3square=np.multiply(SSS3,SSS3);
var1=np.mean(SS1square,axis=0);
var2=np.mean(SS2square,axis=0);
var3=np.mean(SS3square,axis=0);
sigma_asymp_1,_=np.polyfit(np.divide(1,liste_sample),var1,1);
#sigma_asymp_2,_=np.polyfit(np.divide(1,liste_sample),var2,1);
#sigma_asymp_3,_=np.polyfit(np.divide(1,liste_sample),var3,1);
#print(sigma_asymp_1);
#print(sigma_asymp_2);
#print(sigma_asymp_3);

In [ ]:
plt.plot(b_vals,sigma_asymp_1)
np.savetxt("b_vals.dat", b_vals)
np.savetxt("sigma_asymp_1", sigma_asymp_1)
#print(liste_sample)
#print(var1)
#print(SS1square.shape)
#print(SS1square[:,:,5])
#np.loadtxt("A.dat")

In [ ]:
print(sigma_asymp_1);

In [ ]:
print(b_vals)